# 02. 다각도 모델 비교와 개선 실험

v1 모델·뷰 수 비교 → v2 손실함수 개선 → 검증 결과 비교입니다. 재사용하는 학습 엔진은 src/training에 남겨 서버·테스트와 같은 구현을 사용합니다.

실행 결과는 저장하지 않았습니다. 아래 실행 스위치는 기본 `False`이며, 전체 실행만으로 학습·파일 생성·서버 검사가 시작되지 않습니다.

In [ ]:
from pathlib import Path
import os
import sys
import json

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/data').is_dir() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('CQC 저장소 안에서 노트북을 열어 주세요.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('작업 폴더:', ROOT)


## 실험 계획 확인

이 셀은 계획만 메모리에 만들고 학습하지 않습니다.

In [ ]:
from src.training import experiments, improvement_experiments

baseline_plan = experiments.build_plan()
improvement_plan = improvement_experiments.build_plan()
print('기본 실험:', len(baseline_plan), '개 / 개선 실험:', len(improvement_plan), '개')
baseline_plan[:2]


## 학습 실행

GPU 환경에서 필요한 실험 하나만 켜세요. 기존 학습 결과 보존을 위해 새 출력 경로를 지정합니다.

In [ ]:
import subprocess

RUN_BASELINE = False
RUN_IMPROVEMENT = False
for enabled, plan, module, folder in (
    (RUN_BASELINE, baseline_plan, experiments, 'baseline'),
    (RUN_IMPROVEMENT, improvement_plan, improvement_experiments, 'improvement'),
):
    if enabled:
        destination = Path('outputs/notebook-training') / folder
        if destination.exists():
            raise FileExistsError(destination)
        destination.mkdir(parents=True)
        for run in plan:
            run = {**run, 'output_dir': destination / Path(run['output_dir']).name}
            subprocess.run(module.command_for(run, 'cuda'), check=True)


## 기존 결과 비교

이미 저장된 CV 결과만 읽습니다. Test 점수로 모델을 선택하지 않습니다.

In [ ]:
from src.training.report import collect_runs, aggregate_runs, recommendation

RUN_READ_RESULTS = False
if RUN_READ_RESULTS:
    comparison = aggregate_runs(collect_runs(Path('outputs/training')))
    print(json.dumps(recommendation(comparison), ensure_ascii=False, indent=2))
